# Performance Analysis (Sports)

This notebook loads Sports-only inference profiling artifacts for RPG and SASRec from both the shared group artifacts tree and repo-local artifacts. It summarizes the latest available profiling session for each model and creates merged scaling plots for candidate-pool size versus inference time and CUDA runtime memory.

Expected layouts:

```text
<artifact-root>/<model>/perf/sports/<session>/manifest.json
<artifact-root>/<model>/perf/sports/<session>/summaries/profile_summary.csv
<artifact-root>/rpg/perf/sports/<session>/graphs/validate_graph_report.json
```

RPG graph-validation reports are still shown when available; SASRec uses full-sort scoring and has no graph-build phase.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve()
while ROOT.name not in {"RPG", "RPG-uva"} and ROOT.parent != ROOT:
    ROOT = ROOT.parent

ARTIFACT_SOURCES = [
    {"model": "RPG", "root": Path("/projects/prjs2120/groups/group_16/artifacts/rpg/perf/sports")},
    {"model": "RPG", "root": ROOT / "artifacts" / "rpg" / "perf" / "sports"},
    {"model": "SASRec", "root": Path("/projects/prjs2120/groups/group_16/artifacts/sasrec/perf/sports")},
    {"model": "SASRec", "root": ROOT / "artifacts" / "sasrec" / "perf" / "sports"},
]

MODEL_ORDER = ["RPG", "SASRec"]
PLOT_COLORS = {"RPG": "#355C7D", "SASRec": "#2A9D8F"}


In [ ]:
def _read_json(path: Path):
    return json.loads(path.read_text())


def _safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.is_file() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def _decorate_frame(frame: pd.DataFrame, **metadata) -> pd.DataFrame:
    if frame.empty:
        return frame
    result = frame.copy()
    for key, value in metadata.items():
        result[key] = value
    return result


def _model_rank(value: str) -> int:
    return MODEL_ORDER.index(value) if value in MODEL_ORDER else len(MODEL_ORDER)


def load_perf_artifacts():
    manifest_rows = []
    summary_frames = []
    graph_frames = []
    validation_rows = []
    validation_graph_rows = []

    for source in ARTIFACT_SOURCES:
        artifact_root = source["root"]
        source_model = source["model"]
        if not artifact_root.exists():
            continue

        for manifest_path in sorted(artifact_root.rglob("manifest.json")):
            session_root = manifest_path.parent.resolve()
            payload = _read_json(manifest_path)
            summary_path = Path(payload.get("summary_csv", session_root / "summaries" / "profile_summary.csv"))
            raw_path = Path(payload.get("raw_csv", session_root / "raw" / "profile_runs.csv"))
            graph_path = Path(payload.get("graph_csv", session_root / "graphs" / "graph_builds.csv"))

            manifest_rows.append(
                {
                    "model": source_model,
                    "artifact_root": str(artifact_root),
                    "session": session_root.name,
                    "session_root": str(session_root),
                    "manifest_path": str(manifest_path),
                    "summary_path": str(summary_path),
                    "raw_path": str(raw_path),
                    "graph_path": str(graph_path),
                    "has_summary_csv": summary_path.is_file(),
                    "has_raw_csv": raw_path.is_file(),
                    "has_graph_csv": graph_path.is_file(),
                }
            )

            summary_df = _safe_read_csv(summary_path)
            if not summary_df.empty and "method" in summary_df.columns:
                summary_df = summary_df.copy()
                summary_df["method"] = summary_df["method"].replace({"RPG": "RPG", "SASRec": "SASRec"})
            summary_frames.append(
                _decorate_frame(
                    summary_df,
                    model=source_model,
                    artifact_root=str(artifact_root),
                    session=session_root.name,
                    session_root=str(session_root),
                    manifest_path=str(manifest_path),
                    summary_path=str(summary_path),
                    raw_path=str(raw_path),
                    graph_path=str(graph_path),
                )
            )

            graph_df = _safe_read_csv(graph_path)
            graph_frames.append(
                _decorate_frame(
                    graph_df,
                    model=source_model,
                    artifact_root=str(artifact_root),
                    session=session_root.name,
                    session_root=str(session_root),
                    manifest_path=str(manifest_path),
                    graph_path=str(graph_path),
                )
            )

        if source_model != "RPG":
            continue
        for report_path in sorted(artifact_root.rglob("validate_graph_report.json")):
            session_root = report_path.parents[1].resolve()
            payload = _read_json(report_path)
            comparisons = payload.get("comparisons", {})
            validation_rows.append(
                {
                    "model": source_model,
                    "artifact_root": str(artifact_root),
                    "session": session_root.name,
                    "session_root": str(session_root),
                    "report_path": str(report_path),
                    "paper_md_path": str(report_path.with_name("paper.md")),
                    "checkpoint_path": payload.get("checkpoint_path"),
                    "checkpoint_name": None if payload.get("checkpoint_path") is None else Path(payload["checkpoint_path"]).name,
                    "checkpoint_signature": payload.get("checkpoint_signature"),
                    "pool_size": payload.get("pool_size"),
                    "topk": payload.get("topk"),
                    "dense_vs_flat_mean_overlap": comparisons.get("dense_vs_flat", {}).get("mean_overlap_rate"),
                    "dense_vs_flat_exact_match_rate": comparisons.get("dense_vs_flat", {}).get("exact_match_rate"),
                    "flat_vs_hnsw_mean_overlap": comparisons.get("flat_vs_hnsw", {}).get("mean_overlap_rate"),
                    "flat_vs_hnsw_p50_overlap": comparisons.get("flat_vs_hnsw", {}).get("p50_overlap_rate"),
                    "dense_vs_hnsw_mean_overlap": comparisons.get("dense_vs_hnsw", {}).get("mean_overlap_rate"),
                    "dense_vs_hnsw_p50_overlap": comparisons.get("dense_vs_hnsw", {}).get("p50_overlap_rate"),
                }
            )
            for graph_record in payload.get("graph_records", []):
                validation_graph_rows.append(
                    {
                        "model": source_model,
                        "artifact_root": str(artifact_root),
                        "session": session_root.name,
                        "session_root": str(session_root),
                        "report_path": str(report_path),
                        **graph_record,
                    }
                )

    manifest_index = pd.DataFrame(manifest_rows)
    if not manifest_index.empty:
        manifest_index = manifest_index.sort_values(["model", "session", "artifact_root"]).reset_index(drop=True)
    profile_summaries = pd.concat([frame for frame in summary_frames if not frame.empty], ignore_index=True) if any(not frame.empty for frame in summary_frames) else pd.DataFrame()
    graph_builds = pd.concat([frame for frame in graph_frames if not frame.empty], ignore_index=True) if any(not frame.empty for frame in graph_frames) else pd.DataFrame()
    validation_runs = pd.DataFrame(validation_rows)
    validation_graphs = pd.DataFrame(validation_graph_rows)
    return manifest_index, profile_summaries, graph_builds, validation_runs, validation_graphs


In [ ]:
manifest_index, profile_summaries, graph_builds, validation_runs, validation_graphs = load_perf_artifacts()

existing_roots = [f"{source['model']}: {source['root']}" for source in ARTIFACT_SOURCES if source["root"].exists()]
print(
    f"Discovered {len(manifest_index)} profiling sessions and {len(validation_runs)} validation reports across {len(existing_roots)} artifact roots."
)
if existing_roots:
    print("Existing artifact roots:")
    for root in existing_roots:
        print(f"  - {root}")

if manifest_index.empty:
    print("No manifest.json files found under the configured perf artifact roots.")
else:
    display(
        manifest_index[
            [
                "model",
                "session",
                "artifact_root",
                "has_summary_csv",
                "has_raw_csv",
                "has_graph_csv",
                "summary_path",
            ]
        ].sort_values(["model", "session", "artifact_root"], ascending=[True, False, True]).reset_index(drop=True)
    )


## Latest Profiling Sessions

This table keeps the latest discovered profiling session per model and shows the per-pool median latency, CUDA memory, traversal, and `NDCG@10` measurements.


In [ ]:
if profile_summaries.empty:
    print("No profile summary rows available.")
    latest_profile_rows = pd.DataFrame()
else:
    latest_sessions = (
        profile_summaries[["model", "artifact_root", "session", "session_root"]]
        .drop_duplicates()
        .sort_values(["model", "session", "artifact_root"])
        .groupby("model", as_index=False)
        .tail(1)
    )
    latest_profile_rows = profile_summaries.merge(
        latest_sessions[["model", "session_root"]],
        on=["model", "session_root"],
        how="inner",
    ).sort_values(["model", "pool_size"]).reset_index(drop=True)

    display_columns = [
        "model",
        "session",
        "artifact_root",
        "pool_size",
        "graph_backend",
        "epoch_time_s_median",
        "peak_cuda_runtime_delta_allocated_gb_median",
        "peak_cuda_allocated_gb_median",
        "visited_ratio_median",
        "n_visited_items_median",
        "ndcg_at_10_median",
        "checkpoint_signature",
    ]
    existing_columns = [column for column in display_columns if column in latest_profile_rows.columns]
    display(
        latest_profile_rows[existing_columns].style.format(
            {
                "epoch_time_s_median": "{:.2f}",
                "peak_cuda_runtime_delta_allocated_gb_median": "{:.3f}",
                "peak_cuda_allocated_gb_median": "{:.3f}",
                "visited_ratio_median": "{:.4f}",
                "n_visited_items_median": "{:.1f}",
                "ndcg_at_10_median": "{:.5f}",
            }
        )
    )

    if not graph_builds.empty:
        latest_graph_rows = graph_builds.merge(
            latest_sessions[["model", "session_root"]],
            on=["model", "session_root"],
            how="inner",
        ).sort_values(["model", "pool_size"]).reset_index(drop=True)
        if not latest_graph_rows.empty:
            display(Markdown("### RPG graph-build rows"))
            display(latest_graph_rows)


## Merged Paper-Style Scaling Plots

These plots compare the latest RPG and SASRec Sports profiling sessions. Runtime CUDA delta is the paper-faithful memory metric because it subtracts the already-loaded model/graph/embedding baseline.


In [ ]:
if profile_summaries.empty or latest_profile_rows.empty:
    print("No profiling rows available for plotting.")
else:
    plot_rows = latest_profile_rows.sort_values(["model", "pool_size"])
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

    for model, frame in plot_rows.groupby("model"):
        frame = frame.sort_values("pool_size")
        color = PLOT_COLORS.get(model)
        axes[0].plot(frame["pool_size"], frame["epoch_time_s_median"], marker="o", label=model, color=color)
        axes[1].plot(
            frame["pool_size"],
            frame["peak_cuda_runtime_delta_allocated_gb_median"],
            marker="o",
            label=model,
            color=color,
        )
        axes[2].plot(frame["pool_size"], frame["ndcg_at_10_median"], marker="o", label=model, color=color)

    axes[0].set_title("Inference Time")
    axes[0].set_xlabel("Item pool size")
    axes[0].set_ylabel("Epoch time (s)")
    axes[1].set_title("Runtime CUDA Memory")
    axes[1].set_xlabel("Item pool size")
    axes[1].set_ylabel("Peak runtime delta (GiB)")
    axes[2].set_title("NDCG@10 Sanity Check")
    axes[2].set_xlabel("Item pool size")
    axes[2].set_ylabel("Median")

    for axis in axes:
        axis.grid(True, alpha=0.3)
        axis.legend()

    plt.show()


## Latest RPG Graph-Validation Report

If an RPG `validate_graph_report.json` session exists, this section shows the latest overlap summary plus the paper-ready text snippet written by the validation job.


In [ ]:
if validation_runs.empty:
    print("No RPG validate_graph_report.json files found.")
else:
    latest_validation = (
        validation_runs.sort_values(["session", "artifact_root"])
        .tail(1)
        .reset_index(drop=True)
    )
    display(
        latest_validation[
            [
                "session",
                "artifact_root",
                "pool_size",
                "topk",
                "checkpoint_signature",
                "dense_vs_flat_mean_overlap",
                "dense_vs_flat_exact_match_rate",
                "flat_vs_hnsw_mean_overlap",
                "flat_vs_hnsw_p50_overlap",
                "dense_vs_hnsw_mean_overlap",
                "dense_vs_hnsw_p50_overlap",
            ]
        ].style.format(
            {
                "dense_vs_flat_mean_overlap": "{:.4%}",
                "dense_vs_flat_exact_match_rate": "{:.4%}",
                "flat_vs_hnsw_mean_overlap": "{:.4%}",
                "flat_vs_hnsw_p50_overlap": "{:.2%}",
                "dense_vs_hnsw_mean_overlap": "{:.4%}",
                "dense_vs_hnsw_p50_overlap": "{:.2%}",
            }
        )
    )

    latest_validation_row = latest_validation.iloc[0]
    latest_validation_graphs = (
        validation_graphs[validation_graphs["session_root"] == latest_validation_row["session_root"]]
        .sort_values("backend")
        .reset_index(drop=True)
    )
    if not latest_validation_graphs.empty:
        display(latest_validation_graphs)

    paper_md_path = Path(latest_validation_row["paper_md_path"])
    if paper_md_path.is_file():
        display(Markdown("### Paper-ready summary"))
        display(Markdown(paper_md_path.read_text()))
